In [ ]:
# Data Loader for Match Winner Neural Network

#This notebook loads and prepares match data for training the match winner prediction model.
#It loads the match CSV, computes rolling EPA values, and prepares the dataset for training.

#```python
import ast
import json
import math
import random
import re
from pathlib import Path

import numpy as np
import pandas as pd
import torch
from sklearn.metrics import accuracy_score, log_loss, roc_auc_score
from torch import nn
from torch.utils.data import DataLoader, TensorDataset

def set_seed(seed: int = 42) -> None:
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)

set_seed(42)

def resolve_year() -> int:
    raw = input("Enter season year [2026]: ").strip()
    if not raw:
        return 2026
    if not raw.isdigit():
        raise ValueError("Please enter a four-digit season year.")
    return int(raw)

def parse_team_keys(value):
    if isinstance(value, list):
        return [str(item) for item in value]
    if value is None or pd.isna(value):
        return []
    if isinstance(value, str):
        value = value.strip()
        if not value:
            return []
        try:
            parsed = ast.literal_eval(value)
        except (ValueError, SyntaxError):
            return [value]
        if isinstance(parsed, list):
            return [str(item) for item in parsed]
    return []

def team_number_from_key(team_key):
    if isinstance(team_key, str):
        match = re.search(r"(\\d+)", team_key)
        if match:
            return int(match.group(1))
    return int(team_key)

def numeric_cols(df: pd.DataFrame) -> list[str]:
    """Identify all numeric columns including EPA, OPR, cOPR, and Statbotics statistics"""
    cols = []
    exclude_cols = {"team_number", "played_index", "match_number", "set_number", "phase"}
    
    for column in df.columns:
        if column in exclude_cols:
            continue
        # Include all performance metrics
        if (column.startswith("breakdown_") or 
            column.startswith("epa_") or 
            column.startswith("opr_") or 
            column.startswith("copr_") or
            column.startswith("dpr_") or
            column.startswith("ccwm_") or
            column.startswith("statbotics_") or
            column.startswith("win_odds_") or
            column in {"epa", "opr", "copr", "dpr", "ccwm", "matches_played", "statbotics_win_odds"}):
            cols.append(column)
    return cols

def summarize_alliance(group: pd.DataFrame, feature_columns: list[str], prefix: str) -> dict:
    numeric = group[feature_columns].apply(pd.to_numeric, errors="coerce")
    mean = numeric.mean().add_prefix(f"{prefix}_mean_")
    std = numeric.std(ddof=0).fillna(0).add_prefix(f"{prefix}_std_")
    out = {**mean.to_dict(), **std.to_dict()}
    out[f"{prefix}_team_count"] = float(len(group))
    return out

def load_statbotics_data(year: int, phase: str = "before"):
    """Load Statbotics win odds data if available"""
    statbotics_csv = Path(f"frc_matches_{year}_statbotics_{phase}.csv")
    
    if not statbotics_csv.exists():
        print(f"No Statbotics data found at {statbotics_csv}, continuing without it")
        return None
    
    try:
        statbotics_df = pd.read_csv(statbotics_csv, low_memory=False)
        if "phase" in statbotics_df.columns:
            statbotics_df = statbotics_df[statbotics_df["phase"].eq(phase)].copy()
        print(f"Loaded Statbotics data with {len(statbotics_df)} rows")
        return statbotics_df
    except Exception as e:
        print(f"Error loading Statbotics data: {e}, continuing without it")
        return None

def build_dataset(year: int, phase: str = "before"):
    """Build dataset for either 'before' or 'after' match phases"""
    match_csv = Path(f"frc_matches_{year}.csv")
    epa_csv = Path(f"frc_matches_{year}_match_epa_{phase}.csv")
    opr_csv = Path(f"frc_matches_{year}_match_opr_{phase}.csv")
    
    if not match_csv.exists():
        raise FileNotFoundError(f"Missing match CSV: {match_csv}")
    if not epa_csv.exists():
        raise FileNotFoundError(
            f"Missing pre-match EPA CSV: {epa_csv}. Run year_match_EPA_loader.ipynb first."
        )
    if not opr_csv.exists():
        raise FileNotFoundError(
            f"Missing pre-match OPR CSV: {opr_csv}. Run year_match_OPR_loader.ipynb first."
        )

    match_df = pd.read_csv(
        match_csv,
        usecols=["key", "winning_alliance", "played_index", "comp_level", "match_number", "set_number"],
        low_memory=False,
    )
    match_df = match_df[match_df["winning_alliance"].isin(["red", "blue"])].copy()
    match_df = match_df.rename(columns={"key": "match_key"})

    # Load both EPA and OPR data for the specified phase
    epa_df = pd.read_csv(epa_csv, low_memory=False)
    epa_df = epa_df[epa_df["phase"].eq(phase)].copy() if "phase" in epa_df.columns else epa_df.copy()
    
    opr_df = pd.read_csv(opr_csv, low_memory=False)
    opr_df = opr_df[opr_df["phase"].eq(phase)].copy() if "phase" in opr_df.columns else opr_df.copy()
    
    # Merge EPA and OPR data with suffixes to preserve all columns
    combined_df = pd.merge(
        epa_df,
        opr_df,
        on=["match_key", "alliance", "team_key", "team_number"],
        suffixes=("_epa", "_opr"),
        how="outer"  # Use outer join to include all available metrics
    )

    # Load and merge Statbotics data if available
    statbotics_df = load_statbotics_data(year, phase)
    if statbotics_df is not None:
        # Merge Statbotics data, keeping all existing data
        combined_df = pd.merge(
            combined_df,
            statbotics_df,
            on=["match_key", "alliance", "team_key", "team_number"],
            how="left",
            suffixes=("", "_statbotics")
        )
        print(f"Successfully merged Statbotics data")

    # Ensure we capture all performance metrics
    feature_columns = numeric_cols(combined_df)
    if not feature_columns:
        raise RuntimeError("No numeric pre-match feature columns were found.")

    # Debug: Print available columns to verify Statbotics inclusion
    statbotics_features = [col for col in feature_columns if 'statbotics' in col.lower() or 'win_odds' in col.lower()]
    if statbotics_features:
        print(f"Statbotics features included: {statbotics_features}")
    else:
        print("No Statbotics features found in the dataset")

    meta_lookup = match_df.set_index("match_key")
    rows = []
    skipped = 0
    for match_key, group in combined_df.groupby("match_key", sort=False):
        if match_key not in meta_lookup.index:
            skipped += 1
            continue

        meta = meta_lookup.loc[match_key]
        if isinstance(meta, pd.DataFrame):
            meta = meta.iloc[0]
        if pd.isna(meta["winning_alliance"]):
            skipped += 1
            continue

        red = group[group["alliance"].eq("red")]
        blue = group[group["alliance"].eq("blue")]
        if red.empty or blue.empty:
            skipped += 1
            continue

        row = {
            "match_key": match_key,
            "played_index": float(meta["played_index"]),
            "comp_level": str(meta["comp_level"]),
            "match_number": float(meta["match_number"]),
            "set_number": float(meta["set_number"]),
            "red_win": 1.0 if str(meta["winning_alliance"]) == "red" else 0.0,
            "phase": phase,  # Add phase information
        }
        row.update(summarize_alliance(red, feature_columns, "red"))
        row.update(summarize_alliance(blue, feature_columns, "blue"))
        rows.append(row)

    dataset = pd.DataFrame(rows).sort_values(["played_index", "match_number", "set_number"]).reset_index(drop=True)
    if dataset.empty:
        raise RuntimeError("No match rows were assembled for training.")
    return dataset, feature_columns, skipped

# Main execution
year = resolve_year()

# Build datasets for both before and after phases
print("Building 'before' match dataset...")
before_dataset, base_feature_columns, before_skipped = build_dataset(year, "before")

print("\nBuilding 'after' match dataset...")
after_dataset, _, after_skipped = build_dataset(year, "after")

# Add competition level features to both datasets
comp_levels = ["qm", "sf", "f"]
for level in comp_levels:
    before_dataset[f"comp_level_{level}"] = (before_dataset["comp_level"] == level).astype(float)
    after_dataset[f"comp_level_{level}"] = (after_dataset["comp_level"] == level).astype(float)

# Prepare feature columns for both datasets
feature_columns = [
    col
    for col in before_dataset.columns
    if col not in {"match_key", "played_index", "comp_level", "match_number", "set_number", "red_win", "phase"}
]

# Clean both datasets
for dataset in [before_dataset, after_dataset]:
    dataset = dataset[["match_key", "played_index", "comp_level", "match_number", "set_number", "red_win", "phase"] + feature_columns].copy()
    dataset = dataset.replace([np.inf, -np.inf], np.nan).fillna(0.0)

print(f"\nLoaded 'before' dataset with {len(before_dataset):,} matches")
print(f"Loaded 'after' dataset with {len(after_dataset):,} matches")
print(f"Total feature columns: {len(feature_columns)}")

# Count and display various feature types
statbotics_features = [col for col in feature_columns if 'statbotics' in col.lower() or 'win_odds' in col.lower()]
copr_features = [col for col in feature_columns if 'copr' in col.lower() or 'ccwm' in col.lower()]
print(f"Statbotics/win odds features: {len(statbotics_features)}")
print(f"cOPR-related features: {len(copr_features)}")

if statbotics_features:
    print(f"Example Statbotics features: {statbotics_features[:3]}")
if copr_features:
    print(f"Example cOPR features: {copr_features[:3]}")

print(f"Features include EPA, OPR, cOPR, and Statbotics statistics")
print(f"Skipped matches (before): {before_skipped}, (after): {after_skipped}")

# Save both datasets
before_dataset.to_csv(f"match_winner_dataset_{year}_before.csv", index=False)
after_dataset.to_csv(f"match_winner_dataset_{year}_after.csv", index=False)

print(f"\nSaved 'before' dataset to match_winner_dataset_{year}_before.csv")
print(f"Saved 'after' dataset to match_winner_dataset_{year}_after.csv")

# Also save a combined dataset for convenience
combined_dataset = pd.concat([before_dataset, after_dataset], ignore_index=True)
combined_dataset.to_csv(f"match_winner_dataset_{year}.csv", index=False)
print(f"Saved combined dataset to match_winner_dataset_{year}.csv")

before_dataset.head()
